# 03 – Model comparison and optimisation

The baseline (YOLOv8n, 640 px) runs at 11 ms/image on a T4 and 108 ms on CPU, far inside any realistic line budget, but finds only half the packaging errors and none of the twisted meat. So the comparison spends extra compute on **accuracy**:

| Experiment | Change from baseline | Question |
|---|---|---|
| `baseline_yolov8n` | – (already trained in 02) | reference |
| `yolov8s_640` | larger model (11 M vs 3 M parameters) | does capacity help the hard classes? |
| `yolov8n_960` | input upscaled from 720×540 to 960 px | do larger feature maps help small/thin defects? (no new detail is added) |
| `yolov8n_480` | input downscaled to 480 px (optional) | how much accuracy does a cheaper input cost? |

**The model is selected on the validation split**; test results are reported for the selected model, not used to choose it. Section 4 then optimises inference of the selected model (FP16 on GPU, ONNX on CPU) and checks accuracy is unchanged.

Runtime: T4 GPU. Each new training run takes about 30–60 min. Runs are saved to Google Drive and skipped if already finished, so after a disconnect just run again.

In [ ]:
# Get the code and install requirements (safe to re-run)
import os, sys

REPO = "/content/MMA3001-Project-repository"
if not os.path.isdir(REPO):
    !git clone https://github.com/rhal0027/MMA3001-Project-repository.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
!pip install -q -r requirements.txt
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")

from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import importlib
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import userdata
from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops

from rasher_inspection import dataset as ds, evaluate as ev, benchmark as bm, train as tr
for m in (ds, ev, bm, tr):
    importlib.reload(m)

root = ds.download_dataset(userdata.get("ROBOFLOW_API_KEY"), "data/raw/pork-rasher", version=4)
names = ds.load_class_names(root)
data_yaml = tr.write_data_yaml(root)

# Same pass/fail rule as 02_baseline (decided before looking at test results)
MAX_FALSE_PASS = 0.02
THRESHOLDS = np.round(np.arange(0.01, 0.96, 0.01), 2)
corrections = ev.load_missed_defects("results/unlabelled_review_test_seed0.csv")
truth = {"valid": ev.tray_truth(root, "valid"),
         "test": ev.tray_truth(root, "test", corrections)}   # test with the 3 corrected labels

## 1. Train

Same settings as the baseline (50 epochs max, early stopping after 15 without improvement, seed 0, batch 16). `cache="ram"` only speeds up data loading; it does not change what the model learns.

In [ ]:
PROJECT = "/content/drive/MyDrive/rasher_runs"
FORCE_RETRAIN = False      # True re-trains even if a finished run exists

EXPERIMENTS = [
    dict(name="baseline_yolov8n", model="yolov8n.pt", imgsz=640, run=True),   # exists from 02: skipped
    dict(name="yolov8s_640",      model="yolov8s.pt", imgsz=640, run=True),
    dict(name="yolov8n_960",      model="yolov8n.pt", imgsz=960, run=True),
    dict(name="yolov8n_480",      model="yolov8n.pt", imgsz=480, run=False),  # optional: set True if time allows
]

def weights(e):
    return Path(PROJECT) / e["name"] / "weights" / "best.pt"

for e in EXPERIMENTS:
    if not e["run"]:
        continue
    if weights(e).exists() and not FORCE_RETRAIN:
        print("skip (already trained):", e["name"])
        continue
    print("training:", e["name"])
    tr.train_yolo(data_yaml, model=e["model"], imgsz=e["imgsz"], epochs=50, batch=16, seed=0,
                  project=PROJECT, name=e["name"], cache="ram", workers=2)

## 2. Evaluate every trained model

For each model: detection metrics on validation **and** test, the pass/fail operating point (threshold chosen on validation, applied to test), and speed on GPU and CPU. Timing uses a fresh model per device.

In [ ]:
summary, per_class = [], []
test_imgs = ds.list_images(root, "test")

for e in EXPERIMENTS:
    if not weights(e).exists():
        continue
    name, imgsz = e["name"], e["imgsz"]
    print("evaluating:", name)
    model = YOLO(weights(e))

    det = {}
    for split in ("valid", "test"):
        m = model.val(data=str(data_yaml), split="val" if split == "valid" else "test", imgsz=imgsz,
                      batch=16, conf=0.001, project=f"{PROJECT}/{name}", name=f"eval_{split}",
                      exist_ok=True, plots=True, verbose=False)
        det[split] = tr.detection_table(m, names)
        per_class += [dict(experiment=name, split=split, **r) for r in det[split]]

    preds = {s: tr.predict_split(model, root, s, imgsz=imgsz) for s in ("valid", "test")}
    conf = {s: {p.image: p.max_confidence for p in preds[s]} for s in preds}
    op = ev.operating_point(truth["valid"], conf["valid"], truth["test"], conf["test"],
                            MAX_FALSE_PASS, THRESHOLDS)

    timing = {}
    for device, imgs, reps in (("cuda:0", test_imgs, 3), ("cpu", test_imgs[:20], 2)):
        m_d = YOLO(weights(e))
        timing[device] = bm.time_per_item(
            lambda p: m_d.predict(str(p), imgsz=imgsz, device=device, verbose=False), imgs, warmup=5, repeats=reps)

    allrow = {s: det[s][-1] for s in det}
    by_cls = {r["class"]: r for r in det["test"]}
    epochs = len(pd.read_csv(Path(PROJECT) / name / "results.csv"))
    summary.append(dict(
        experiment=name, model=e["model"].removesuffix(".pt"), imgsz=imgsz, epochs_trained=epochs,
        params_M=round(sum(p.numel() for p in model.model.parameters()) / 1e6, 2),
        gflops=round(get_flops(model.model, imgsz), 1),
        val_mAP50=allrow["valid"]["mAP50"], val_mAP50_95=allrow["valid"]["mAP50_95"],
        test_mAP50=allrow["test"]["mAP50"], test_mAP50_95=allrow["test"]["mAP50_95"],
        test_unsealed_mAP50=by_cls.get("unsealed", {}).get("mAP50"),
        test_packaging_mAP50=by_cls.get("packaging-error", {}).get("mAP50"),
        test_packaging_recall=by_cls.get("packaging-error", {}).get("recall"),
        threshold=op["threshold"], threshold_met_limit=op["met_limit"],
        val_false_pass=op["valid"]["false_pass_rate"], test_false_pass=op["test"]["false_pass_rate"],
        val_false_fail=op["valid"]["false_fail_rate"], test_false_fail=op["test"]["false_fail_rate"],
        gpu_ms=timing["cuda:0"]["median_ms"], cpu_ms=timing["cpu"]["median_ms"],
    ))

cmp = pd.DataFrame(summary).set_index("experiment").round(3)
cmp.to_csv("results/compare_summary.csv")
pd.DataFrame(per_class).round(3).to_csv("results/compare_per_class.csv", index=False)
cmp.T

In [ ]:
# Accuracy vs speed, with validation (used for selection) and test (reported) side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, xcol, xlabel in ((axes[0], "gpu_ms", "GPU ms / image (T4, median)"),
                         (axes[1], "cpu_ms", "CPU ms / image (median)")):
    ax.scatter(cmp[xcol], cmp["val_mAP50_95"], marker="o", label="validation")
    ax.scatter(cmp[xcol], cmp["test_mAP50_95"], marker="x", label="test")
    for exp, r in cmp.iterrows():
        ax.annotate(exp, (r[xcol], r["val_mAP50_95"]), textcoords="offset points", xytext=(5, 5), fontsize=8)
    ax.set_xlabel(xlabel); ax.set_ylabel("mAP50-95"); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle("Accuracy vs inference time")
fig.tight_layout()
fig.savefig("results/compare_accuracy_vs_speed.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Select a model (on validation only)

**Time budget:** packing lines run at about 50–90 packs per minute. Designing for the fast end, 90 packs/min gives 60,000 / 90 ≈ 667 ms per tray. Keeping a 10× margin for image capture, the reject actuator and the rest of the controller gives **≈ 67 ms per image** on the target hardware.

**Rule:** highest validation mAP50-95 among models within the budget. If two models are within 0.01 mAP50-95, the cheaper one (fewer GFLOPs) wins. `TARGET = "gpu_ms"` assumes the line has a GPU; set `"cpu_ms"` to check a CPU-only installation (section 4 tests whether ONNX brings the CPU time under budget).


In [ ]:
PACKS_PER_MIN, MARGIN = 90, 10
BUDGET_MS = 60_000 / PACKS_PER_MIN / MARGIN     # ≈ 67 ms per image
TARGET = "gpu_ms"
print(f"Budget: {BUDGET_MS:.0f} ms per image on {TARGET[:3].upper()}")

ok = cmp[cmp[TARGET] <= BUDGET_MS]
best_val = ok["val_mAP50_95"].max()
contenders = ok[ok["val_mAP50_95"] >= best_val - 0.01]
SELECTED = contenders["gflops"].idxmin()
print("Within budget:", list(ok.index))
print("Selected:", SELECTED)
cmp.loc[[SELECTED]].T

## 4. Inference optimisation of the selected model

Two changes that need no retraining, each checked for speed **and** accuracy (an optimisation that changes the answers is not free):

- **FP16 on GPU**: half-precision arithmetic; T4 tensor cores are built for it.
- **ONNX Runtime on CPU**: exported graph with fused operations, run by an optimised CPU engine instead of PyTorch.

In [ ]:
!pip install -q onnx onnxruntime onnxslim

sel = next(e for e in EXPERIMENTS if e["name"] == SELECTED)
imgsz, w = sel["imgsz"], weights(sel)
onnx_path = YOLO(w).export(format="onnx", imgsz=imgsz, simplify=True)

variants = [  # (label, weights, device, half)
    ("PyTorch FP32 GPU", w,         "cuda:0", False),
    ("PyTorch FP16 GPU", w,         "cuda:0", True),
    ("PyTorch FP32 CPU", w,         "cpu",    False),
    ("ONNX FP32 CPU",    onnx_path, "cpu",    False),
]
opt = []
for label, wt, device, half in variants:
    m_v = YOLO(wt, task="detect")
    imgs = test_imgs if device != "cpu" else test_imgs[:20]
    t = bm.time_per_item(lambda p: m_v.predict(str(p), imgsz=imgsz, device=device, half=half, verbose=False),
                         imgs, warmup=5, repeats=3 if device != "cpu" else 2)
    mv = m_v.val(data=str(data_yaml), split="test", imgsz=imgsz, batch=1 if device == "cpu" else 16,
                 device=device, half=half, conf=0.001, plots=False, verbose=False)
    opt.append(dict(variant=label, median_ms=t["median_ms"], p95_ms=t["p95_ms"],
                    trays_per_min=round(bm.trays_per_minute(t["median_ms"])),
                    test_mAP50=mv.box.map50, test_mAP50_95=mv.box.map))

opt_df = pd.DataFrame(opt).set_index("variant").round(3)
for base, new in (("PyTorch FP32 GPU", "PyTorch FP16 GPU"), ("PyTorch FP32 CPU", "ONNX FP32 CPU")):
    opt_df.loc[new, "speedup"] = round(opt_df.loc[base, "median_ms"] / opt_df.loc[new, "median_ms"], 2)
    opt_df.loc[new, "mAP50_95_change"] = round(opt_df.loc[new, "test_mAP50_95"] - opt_df.loc[base, "test_mAP50_95"], 4)
opt_df["hardware"] = [bm.environment()["gpu"] if "GPU" in v else bm.environment()["cpu"] for v in opt_df.index]
opt_df["within_budget"] = opt_df["median_ms"] <= BUDGET_MS
opt_df.to_csv(f"results/optimise_{SELECTED}.csv")
opt_df

## 5. Save results

Download the zip and attach it in the chat. Weights and full Ultralytics outputs stay in Google Drive (`rasher_runs/`).

In [ ]:
!cd results && zip -q ../results_compare.zip compare_* optimise_*
from google.colab import files
files.download("results_compare.zip")